# Similarity-stratified invariant kinase profile coverage

This notebook evaluates a new model derived from the completed PKIS2 mechanism audit. It replaces unsupported fine-grained molecule-to-profile weights with a deterministic group average inside five chemical-similarity strata, then tests a reference-only pseudo-holdout gate that can abstain to marginal ranking.

**Statistical status:** exploratory/post hoc model development. It does not revise the failed confirmatory Klaeger result.

**Scope:** retrospective kinase-panel first-activity and binding-liability prioritization. No output establishes cellular engagement, selectivity, toxicity, efficacy, clinical safety, or therapeutic suitability.

**Compute:** CPU-only, no paid API, no model training. A GPU is not used.

In [ ]:
# Frozen controls. Leave SMOKE_TEST=False for the complete study.
SMOKE_TEST = False
WORKERS = 8
BOOTSTRAP_REPLICATES = 10_000
SIGN_FLIP_PERMUTATIONS = 100_000
EXPECTED_BUNDLE = 'invariant_coverage_colab_bundle.zip'
PKIS2_URL = 'https://doi.org/10.1371/journal.pone.0181585.s004'
PKIS2_SHA256 = '48ead22a1f860cd0d5096fa87d5acd329f722fe8d65e693bb0be682a333e2a2c'
print({
    'smoke_test': SMOKE_TEST,
    'worker_processes': WORKERS,
    'bootstrap': BOOTSTRAP_REPLICATES,
    'permutations': SIGN_FLIP_PERMUTATIONS,
    'gpu_required': False,
    'paid_api_cost_usd': 0,
})

## 1. Upload and verify the frozen bundle

Upload `invariant_coverage_colab_bundle.zip`. The compact bundle includes the licensed derived Klaeger/ChEMBL records. PKIS2 is downloaded separately from PLOS.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, urllib.request, zipfile

uploaded = files.upload()
assert EXPECTED_BUNDLE in uploaded, f'Upload {EXPECTED_BUNDLE}'
archive = Path('/content') / EXPECTED_BUNDLE
archive.write_bytes(uploaded[EXPECTED_BUNDLE])
with zipfile.ZipFile(archive) as zf:
    assert zf.testzip() is None, 'Corrupt archive member'
    zf.extractall('/content')
project = Path('/content/counterscreen_active_search')
assert project.is_dir()
freeze = subprocess.run(
    ['sha256sum', '-c', 'INVARIANT_COVERAGE_FREEZE.sha256'],
    cwd=project, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(freeze.stdout)
assert freeze.returncode == 0, 'Frozen files failed verification'

## 2. Install exact dependencies and run deterministic tests

In [ ]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(project / 'requirements_method_colab.txt')],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(install.stdout[-8000:])
assert install.returncode == 0
for test_file in ['src/test_compiled_coverage.py', 'src/test_invariant_coverage.py']:
    tests = subprocess.run(
        [sys.executable, test_file], cwd=project, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(tests.stdout)
    assert tests.returncode == 0, f'Tests failed: {test_file}'
validation = subprocess.run(
    [sys.executable, 'src/validate_dataset.py', '--data-dir', 'data/derived',
     '--report', '/content/invariant_dataset_validation.json'],
    cwd=project, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(validation.stdout)
assert validation.returncode == 0 and '"status": "PASS"' in validation.stdout

## 3. Download and verify PKIS2

The exact Drewry et al. PLOS ONE S4 workbook is distributed under CC BY 4.0. Its single-concentration measurements remain separate from Klaeger `Kd` values.

In [ ]:
pkis2_path = Path('/content/pkis2_s4.xlsx')
urllib.request.urlretrieve(PKIS2_URL, pkis2_path)
observed_hash = hashlib.sha256(pkis2_path.read_bytes()).hexdigest()
print('PKIS2 SHA-256:', observed_hash)
assert observed_hash == PKIS2_SHA256

## 4. Run M0–M4 across all eight frozen conditions

M3 is the primary new invariant model. M4 uses only nested reference pseudo-holdouts to decide whether to use M3 or abstain to invariant marginal ranking. The PKIS2 chemotype condition additionally receives 50 fixed 80% reference-subsample stability checks per compound.

In [ ]:
output_dir = project / ('invariant_coverage_smoke_output' if SMOKE_TEST else 'invariant_coverage_output')
if output_dir.exists():
    shutil.rmtree(output_dir)
command = [
    sys.executable, 'src/run_invariant_coverage.py',
    '--data-dir', 'data/derived',
    '--pkis2-xlsx', str(pkis2_path),
    '--output-dir', output_dir.name,
    '--bootstrap', str(BOOTSTRAP_REPLICATES),
    '--permutations', str(SIGN_FLIP_PERMUTATIONS),
    '--workers', str(WORKERS),
]
if SMOKE_TEST:
    command.append('--smoke-test')
run_environment = dict(os.environ)
for variable in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    run_environment[variable] = '1'
started = time.time()
with subprocess.Popen(
    command, cwd=project, text=True, env=run_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
) as process:
    captured = []
    for line in process.stdout:
        captured.append(line)
        print(line, end='')
    return_code = process.wait()
output_dir.mkdir(exist_ok=True)
(output_dir / 'console.log').write_text(''.join(captured), encoding='utf-8')
assert return_code == 0, 'Experiment failed; inspect console.log'
print(f'Wall time: {time.time() - started:.1f} seconds')

## 5. Inspect the frozen decisions

No success threshold is changed after this output. Smoke-test decisions are not scientific results.

In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text(encoding='utf-8'))
print('Status:', summary['status'])
print('\nFROZEN SUCCESS CRITERIA')
for name, value in summary['frozen_success_criteria'].items():
    print(f'  {name}: {value}')
print('\nCONDITION EFFECTS')
for condition, results in summary['comparisons'].items():
    original = results['original_coverage_minus_original_marginal']
    invariant = results['invariant_coverage_minus_invariant_marginal']
    gated = results['selective_invariant_minus_invariant_marginal']
    print(condition)
    print('  original:', original['estimate'], original['ci95'])
    print('  invariant:', invariant['estimate'], invariant['ci95'])
    print('  selective:', gated['estimate'], gated['ci95'],
          'large delays:', gated['large_delay_ge_10'])
print('\nSTABILITY:', summary['stability'])
print('\nGATE COVERAGE RATES')
for condition, gate in summary['gates'].items():
    print(condition, gate['coverage_rate'])

In [ ]:
from IPython.display import display, Image
for figure in sorted((output_dir / 'figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))

## 6. Verify and download every artifact

In [ ]:
required = [
    'summary.json', 'comparison_summary.csv', 'case_metrics.csv.gz',
    'gate_diagnostics.csv.gz', 'stability_diagnostics.csv.gz',
    'environment.json', 'output_manifest.json', 'console.log',
    'INVARIANT_COVERAGE_PROTOCOL.md',
    'figures/condition_effects.png', 'figures/stability_comparison.png',
    'figures/chemotype_tail_risk.png',
]
missing = [name for name in required if not (output_dir / name).is_file()]
assert not missing, f'Missing artifacts: {missing}'
manifest_rows = []
for artifact in sorted(output_dir.rglob('*')):
    if artifact.is_file() and artifact.name != 'output_manifest.json':
        manifest_rows.append({
            'path': artifact.relative_to(output_dir).as_posix(),
            'bytes': artifact.stat().st_size,
            'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest(),
        })
(output_dir / 'output_manifest.json').write_text(
    json.dumps({'files': manifest_rows}, indent=2, sort_keys=True) + '\n',
    encoding='utf-8'
)
result_base = Path('/content/invariant_coverage_results')
result_zip = Path(shutil.make_archive(str(result_base), 'zip', root_dir=output_dir))
print('Result SHA-256:', hashlib.sha256(result_zip.read_bytes()).hexdigest())
files.download(str(result_zip))